# Run smoke test training and inference

Purpose
-------

- Opitnonally launch the smoke training helper (dry-run by default);
- Verify environment dependencies;
- Run a quick inference example using a prefuilt TTS model and save WAVs.

In [ ]:
from pathlib import Path
import subprocess
import json
from statistics import mean
from typing import List

project_root = Path(".")
manifest_path = project_root / "data/processed/manifests/smoke_manifest.jsonl"
config_path = project_root / "configs/smoke_vits.yaml"
run_script_path = project_root / "scripts/run_finetune_smoke.bat"
outputs_dir = project_root / "outputs/smoke_vits"

run_training = False
training_timeout_seconds = 300

## Check environment and dependencies

- Verifies whether required packages are importable and prints pip install suggestions.

In [ ]:
missing_packages: List[str] = []
try:
    from TTS.api import TTS  # type: ignore
except Exception:
    missing_packages.append("TTS")

try:
    import librosa  # type: ignore
except Exception:
    missing_packages.append("librosa")

try:
    import soundfile as sf  # type: ignore
except Exception:
    missing_packages.append("soundfile")

try:
    import scipy  # type: ignore
except Exception:
    missing_packages.append("scipy")

if missing_packages:
    print("Missing packages:", missing_packages)
    print("Install with: pip install " + " ".join(missing_packages))
else:
    print("All optional dependencies appear available.")

## Run trainer helper (optional)

- Only run if you understand the helper will start training. The helper script is the `.bat` created by notebook 06.

In [ ]:
def run_training_helper(script_path: Path, timeout_seconds: int = 300) -> int:
    if not script_path.exists():
        raise FileNotFoundError(f"Run script not found: {script_path}")

    proc = subprocess.Popen(str(script_path), shell=True)

    try:
        proc.wait(timeout=timeout_seconds)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise TimeoutError(f"Training helper exceeded timeout of {timeout_seconds} seconds")

    return proc.returncode

if run_training:
    print("Launching training helper:", run_script_path)
    try:
        run_return = run_training_helper(run_script_path, timeout_seconds=training_timeout_seconds)
        print("Training helper exited with return code", run_return)
    except Exception as error:
        print("Training helper failed:", error)
else:
    print("run_training is False 	6 not launching the training helper (dry run).")

## Inference example

- Loads a prebuilt Coqui TTS model and writes short WAVs to `outputs/smoke_vits/inference/`;
- Replace `model_name` with your checkpoint path to test a fine-tuned model.

In [ ]:
inference_output_dir = outputs_dir / "inference"
inference_output_dir.mkdir(parents=True, exist_ok=True)

sample_sentences = [
    "Olá, este é um teste.",
    "Este é apenas um teste."
]

model_name = "tts_models/pt/cv/vits"

try:
    print("Loading TTS model:", model_name)
    tts = TTS(model_name=model_name)
    sample_rate = getattr(getattr(tts, "synthesizer", None), "output_sample_rate", 22050)
    saved_files = []

    for idx, text in enumerate(sample_sentences, start=1):
        wav_array = tts.tts(text)
        output_path = inference_output_dir / f"sample_{idx}.wav"
        sf.write(str(output_path), wav_array, sample_rate)
        saved_files.append(output_path)
        print("Wrote:", output_path)

    print("Inference complete. Written files:", saved_files)

except Exception as error:
    print("Inference failed:", error)